In [ ]:
from matplotlib.pyplot import figure, show, subplots
import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
import random
from scipy.optimize import curve_fit
import csv
from scipy.stats import mannwhitneyu
from scipy.interpolate import RegularGridInterpolator
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.lines import Line2D
from tabulate import tabulate
import os

from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import BayesianRidge
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import fisher_exact
import statsmodels.api as sm
import pymc as pm
from scipy.optimize import curve_fit

import ReadPlanetPopulation as RPP
import subprocess

#import yaml
from ruamel.yaml import YAML
yaml = YAML()
yaml.preserve_quotes = True  # Keep quotes if they exist
yaml.indent(mapping=2, sequence=4, offset=2)  # Preserve indentation
yaml.representer.add_representer(
    float, lambda dumper, data: dumper.represent_scalar("tag:yaml.org,2002:float", str(data))
)

with open("../config/masterconfig.yaml", "r") as file:
    config = yaml.load(file) or {}
PPOP_rootpath = config['PPOP_rootpath']
REPO_rootpath = config['REPO_path']

In [ ]:
# Bin edges (Gyr)
edges = np.array([
    0.0,1.00, 1.25, 1.50, 1.75, 2.00, 2.25, 2.50, 2.75,
    3.00, 3.25, 3.50, 3.75, 4.00, 4.25, 4.50, 4.75,
    5.00, 5.25, 5.50, 5.75, 6.00, 6.25, 6.50, 6.75,
    7.00, 7.25, 7.50, 7.75
])

# Histogram heights
heights = np.array([
    4.38,4.38, 4.11, 5.38, 2.20, 3.43, 3.93, 5.58, 6.83,
    8.31, 7.49, 8.54, 8.16, 10.46, 12.71, 12.67,
    15.21, 16.39, 14.82, 13.46, 13.88, 10.72,
    5.75, 2.56, 0.77, 0.42, 0.35, 0.14
])
print(len(edges))
print(len(heights))
# Bin widths
widths = np.diff(edges)

# Probability mass in each bin
weights = heights * widths
weights /= weights.sum()

# Cumulative distribution
cdf = np.cumsum(weights)

def random_age(N):
    """
    Draw N ages from the digitized histogram.
    """
    u = np.random.random(N)

    # Choose bins according to probability mass
    idx = np.searchsorted(cdf, u)

    # Uniformly sample within selected bin
    ages = edges[idx] + np.random.random(N) * widths[idx]

    return ages


# --------------------------------------------------
# Example
# --------------------------------------------------

N = 1000000
ages = random_age(N)

# # Plot sampled histogram
# plt.figure(figsize=(8,6))

# plt.hist(
#     ages,
#     bins=edges,
#     density=True,
#     alpha=0.5,
#     label="Sampled ages"
# )

# Plot target distribution
target_density = heights / np.sum(heights * widths)

plt.stairs(
    target_density,
    edges,
    linewidth=2,
    label="Target distribution"
)
plt.vlines(np.median(ages),ymin=0,ymax=0.3,color='black')

plt.xlabel("Stellar age (Gyr)")
plt.ylabel("Probability density")
plt.legend()

plt.show()


In [ ]:
# define redox state function

def redox_linear(t):
    x = [0,8]
    y = [-4,4]
    return np.interp(t,x,y)

def redox_constant(t):
    return t*0

def redox_constant_2(t):
    return t*0+2

def redox_inverse(t):
    return np.maximum(-(1/t)+4,-4.0)


time = np.linspace(1e-20,8,1000)
fig = figure(figsize=(8,8))
frame1 = fig.add_subplot(2,2,1)
frame1.plot(time,redox_linear(time))
frame1.set_xlim(0,8)
frame1.set_ylim(-4,4)
frame1.set_xlabel('stellar age (Gyr)')
frame1.set_ylabel('$f$O$_2$, $\Delta$ IW')

frame2 = fig.add_subplot(2,2,2)
frame2.plot(time,redox_inverse(time))
frame2.set_xlim(0,8)
frame2.set_ylim(-4,4)
frame2.set_xlabel('stellar age (Gyr)')
frame2.set_ylabel('fo2 (IW)')

frame3 = fig.add_subplot(2,2,3)
frame3.plot(time,redox_constant(time))
frame3.set_xlim(0,8)
frame3.set_ylim(-4,4)
frame3.set_xlabel('stellar age (Gyr)')
frame3.set_ylabel('fo2 (IW)')

frame4 = fig.add_subplot(2,2,4)
frame4.plot(time,redox_constant_2(time))
frame4.set_xlim(0,8)
frame4.set_ylim(-4,4)
frame4.set_xlabel('Stellar age (Gyr)')
frame4.set_ylabel('$f$O$_2$, $\Delta$ IW')

fig.suptitle('Different artificial redox trends with respect to stellar age')
show()


In [ ]:
def ppop_Nuniverses( N_universes,filepath="../config/P-pop.py", line_number=116):
    with open(filepath, 'r') as f:
        lines = f.readlines()

    # line_number is 1-based (like in editors)
    lines[line_number - 1] = "Nuniverses = " + str(N_universes) + ' # int' + '\n'

    with open(filepath, 'w') as f:
        f.writelines(lines)

def mixingratios_noise(mixingratios,noise=10.0):
    if noise == 0.0:
        return mixingratios.copy()
    else:
        log_noise = np.random.normal(0, noise, len(mixingratios))
        noiselist = 10**log_noise
        noiseMR = noiselist * mixingratios
        return noiseMR / np.sum(noiseMR)
    
def plot_single_piechart(atmosphere_comp):
    volatile_labels = ['N2','CO','H2','CH4','H2S','SO2','H2O','CO2','NH3','OCS']
    volatile_colours = ["gray","magenta","red","green","pink","orange","blue","cyan","purple","navy"]
    mpl.rcParams['figure.dpi'] = 100
    rasterplot = figure(figsize=(8,6))
    frame = rasterplot.add_subplot(1,1,1)
    frame.pie(atmosphere_comp, labels=None,
         colors=volatile_colours)
    show()

In [ ]:
# import retrieval data
# use known/real data for now
data_location = "../output/AGNI_atmospherecomposition_normalized"
data = np.genfromtxt(data_location,skip_header=1,delimiter=',')
volatile_labels = np.genfromtxt(data_location,delimiter=',',dtype='str',skip_footer=len(data))

# use only molecules that will actually be in the retrieval
keep_data = ['fo2','C/H','N2','CO','H2','CH4','H2S','SO2','H2O','CO2','NH3','OCS']
keep_data_index = []
for i in keep_data:
    keep_data_index.append(list(volatile_labels).index(i))
volatile_labels = volatile_labels[keep_data_index]
atmosphere_data = [data[i][keep_data_index] for i in range(len(data))]

#normalize
new_data = []
for i in atmosphere_data:
    coords = [i[0],i[1]]
    normal_data = i[2:]/sum(i[2:])
    for j in normal_data:
        coords.append(j)
    new_data.append(coords)
atmosphere_data = new_data

atmosphere_dict = {(row[0], row[1]): row[2:] for row in atmosphere_data}

# train model

learningdata = np.array(atmosphere_data)[:,2:]
learninglabels=['reduced','reduced','reduced','reduced','reduced','reduced','oxidized','oxidized','oxidized','oxidized','oxidized','oxidized']

# feature scaling
learningdata_log = np.log10(learningdata+1e-20)
scaler = StandardScaler()
learningdata_log_scaled = scaler.fit_transform(learningdata_log)

# logisticregression
clf_LR = LogisticRegression()
clf_LR = clf_LR.fit(learningdata_log_scaled, learninglabels)

y_pred = clf_LR.predict(np.array(atmosphere_data)[-1][2:].reshape(1, -1))
print(y_pred)

# coefs = clf_LR.coef_[0]
# plt.barh(volatile_labels[2:], -1*coefs)
# plt.axvline(0)
# plt.xlabel("Effect on log-odds")
# plt.title("Feature influence")
# plt.show()

# ridge

print("-------")
clf_ridge = BayesianRidge()
learninglabels = np.array(atmosphere_data)[:,0]
print(learninglabels)
clf_ridge.fit(learningdata, learninglabels)

ridge_pred = []
ridge_std = []
for i in range(len(atmosphere_data)):
    testdata = np.array(atmosphere_data)[i][2:].reshape(1, -1)
    y_pred, std = clf_ridge.predict(testdata,return_std=True)
    print(y_pred,std)
    ridge_pred.append(y_pred[0])
    ridge_std.append(std[0])

weights = clf_ridge.coef_

plt.figure(figsize=(8,6))
plt.barh(volatile_labels[2:], weights)
plt.axvline(0)  # zero reference
plt.xlabel("Weight")
plt.title("Molecular influence on FO2 prediction")
plt.show()


In [ ]:
#print(atmosphere_data)
# pie charts for AGNI data
volatile_labels = ['N2','CO','H2','CH4','H2S','SO2','H2O','CO2','NH3','OCS']
volatile_colours = ["gray","magenta","red","green","pink","orange","blue","cyan","purple","navy"]


def savefigures(data,filename,plot_title):
   mpl.rcParams['figure.dpi'] = 800
   rasterplot = figure(figsize=(8,6))

   # setup custom order for rasterplot, since the subplots go left->right, up->down
   # but we need to go down->up, left->right
   custom_order = [9,5,1,10,6,2,11,7,3,12,8,4]

   # do plots
   for i in range(len(atmosphere_data)):
      frame = rasterplot.add_subplot(3,4,custom_order[i])
      frame.pie(atmosphere_data[i][2:], labels=None,
         colors=volatile_colours)

   # add axes to the plot (I feel like I over-engineered this)
   master_ax = rasterplot.add_axes([0.1, 0.1, 0.8, 0.8])
   master_ax.patch.set_alpha(0)
   master_ax.set_zorder(-10)

   master_ax.spines['top'].set_visible(False)
   master_ax.spines['right'].set_visible(False)
   master_ax.spines['left'].set_linewidth(1.5)
   master_ax.spines['bottom'].set_linewidth(1.5)

   y_centers = [0.83, 0.50, 0.17] 
   master_ax.set_ylim(0, 1)
   master_ax.set_yticks(y_centers)
   master_ax.set_yticklabels(["5", "1", "0.1"])

   x_centers = [0.135, 0.365, 0.635, 0.865]
   master_ax.set_xlim(0, 1)
   master_ax.set_xticks(x_centers)
   master_ax.set_xticklabels(["-4", "-1.3", "1.3", "4"])

   master_ax.tick_params(labelsize=12)

   # add legend
   custom_lines = []
   for i in volatile_colours:
      custom_lines.append(Line2D([0], [0], color=i, lw=4))

   rasterplot.legend(custom_lines,volatile_labels,bbox_to_anchor=(1.05, 0.9))

   # axes labels
   rasterplot.supxlabel('Mantle $f$O$_2$, $\Delta$ IW')
   rasterplot.supylabel('Mantle C/H mass ratio')
   #rasterplot.suptitle(plot_title)

   #filepath = "../output/figures/" + filename
   #plt.savefig(filepath, bbox_inches="tight")

savefigures(atmosphere_data,'AGNI_3dplot_molefraction.png',"Molecular contents of volcanic gas parcel (mole fractions)")

In [ ]:
fig = figure(figsize=(8,6))
frame = fig.add_subplot(1,1,1)
frame.scatter(learninglabels,learninglabels,color='blue',label='Learning data')
print(learninglabels)
print(ridge_pred)
print(ridge_std)
frame.errorbar(learninglabels[::3]+0.1,ridge_pred[::3],yerr=ridge_std[::3],fmt='o',color='red',label='Model applied to learning data')
frame.errorbar(learninglabels[1::3]-0.1,ridge_pred[1::3],yerr=ridge_std[1::3],fmt='o',color='red')
frame.errorbar(learninglabels[2::3],ridge_pred[2::3],yerr=ridge_std[2::3],fmt='o',color='red')
frame.set_xlabel("True fo2")
frame.set_ylabel("Predicted fo2")


for i in np.linspace(-4.0,4.0,20):
    CH = 0.1
    mixingratios = np.array(atmosphere_interpolator(atmosphere_dict,[i,CH]))
    y_pred, std = clf_ridge.predict(mixingratios.reshape(1, -1),return_std=True)
    if i == -4.0:
        frame.errorbar(i,y_pred,yerr=std,color="green",fmt='o',label='Model applied to interpolated data')
    else:
        frame.errorbar(i,y_pred,yerr=std,color="green",fmt='o')

frame.legend(loc='upper left')
show()

In [ ]:
# import retrieval data

done = []
data_location = "/dataserver/users/formingworlds/borgmann/ARCiS_retrievals/"
atmosphere_folders = os.listdir(data_location)
for i in atmosphere_folders:
    if os.path.isfile(data_location+i+"/retrieval"):
        if i[-7:] == 'highobv':
            continue
        else:
            done.append(i)


volatile_labels = np.genfromtxt(data_location+done[0]+"/retrieval",usecols=0,dtype='str')[1:11]
print(volatile_labels)

atmosphere_data = []
for i in range(len(done)):
    medians = np.genfromtxt(data_location+done[i]+"/retrieval",usecols=1)[1:11]
    medians = medians/sum(medians)
    atmosphere_data.append([[float(j) for j in done[i].split("_")],medians])

atmosphere_dict = {(x,y,z,t): arr for [x,y,z,t], arr in atmosphere_data}

print(atmosphere_dict[(-1.33,5.0,10.0,350.0)])
#plot_single_piechart(atmosphere_dict[(-4.0,0.1,20.0,350.0)])
#print(atmosphere_data)


In [ ]:
selected_obs_time = 350.0

filtered_dict = {
    (fo2, CH, dist): values
    for (fo2, CH, dist, obs_time), values in atmosphere_dict.items()
    if obs_time == selected_obs_time
}
fo2_grid = sorted({k[0] for k in filtered_dict})
CH_grid = sorted({k[1] for k in filtered_dict})
dist_grid = sorted({k[2] for k in filtered_dict})

n_species = len(next(iter(filtered_dict.values())))

values = np.zeros((
    len(fo2_grid),
    len(CH_grid),
    len(dist_grid),
    n_species
))

fo2_idx = {v: i for i, v in enumerate(fo2_grid)}
CH_idx = {v: i for i, v in enumerate(CH_grid)}
dist_idx = {v: i for i, v in enumerate(dist_grid)}

for (fo2, CH, dist), abundances in filtered_dict.items():
    i = fo2_idx[fo2]
    j = CH_idx[CH]
    k = dist_idx[dist]

    values[i, j, k, :] = abundances

interp = RegularGridInterpolator(
    (fo2_grid, CH_grid, dist_grid),
    values
)
result = interp([-4.0, 1.0, 17.5])
print(result[0])
plot_single_piechart(result[0])

In [64]:
# survey loop

def LIFE_survey(N_iterations=10,N_planets=100,CH=0.1,redoxfunc="linear",classifier="binary",stat_test='FisherExact',noise=0.0):
    if classifier == "binary":
        # define binary age bins
        age_pop = random_age(10000)
        agebin_condition = np.median(age_pop)
        #agebin_condition = 2.0
        age_old = agebin_condition + 0.25*np.max(age_pop)
        age_young = agebin_condition - 0.25*np.max(age_pop)

    # run p_pop once, generating N_iterations amount of universes
    
    # # change Nuniverses in p_pop
    # ppop_Nuniverses(N_universes=N_iterations)

    # # write config file to p-pop repo
    # prompt = "cp -f " + "../config/P-pop.py " + PPOP_rootpath
    # subprocess.run(prompt, shell=True)

    # # run p-pop
    # prompt = "python3 P-pop.py"
    # subprocess.run(prompt,cwd=PPOP_rootpath, shell=True)

    # # get population data file
    # prompt = "cp -f " + PPOP_rootpath + "/planet_population.txt " + "../output"
    # subprocess.run(prompt, shell=True)

    
    # start loop
    N_it = -1
    succes_count = 0
    fail_count = 0
    while N_it < N_iterations-1:
        N_it += 1

        # get p-pop data
        data = np.genfromtxt("../output/planet_population.txt",skip_header=2,usecols=(0,20,24))
        select_universe = np.where([data[i][0] == N_it for i in range(len(data))])
        data = data[:,1:3][select_universe]

        titles = np.genfromtxt("../output/planet_population.txt",dtype='str',usecols=(20,24))[1]
        data_sorted = np.array(sorted(data,key=lambda x: x[1]))

        # pick planets
        target_planets = [] # distances only
        # pick random planet of 100 closest stars
        for i in range(len(data_sorted[:,0])):
            if i == 0:
                current_star = data_sorted[i][0]
                target_planets.append(data_sorted[i][1])
            else:
                if current_star == data_sorted[i][0]:
                    continue
                else:
                    current_star = data_sorted[i][0]
                    target_planets.append(data_sorted[i][1])
            if len(target_planets) == N_planets:
                break
        
        target_ages = []
        target_fo2 = []
        target_atm_contents = []
        target_atm_contents_scaled = []
        for i in range(len(target_planets)):
            planet_age = random_age(1)[0]
            if redoxfunc == "linear":
                fo2 = redox_linear(planet_age)
                target_fo2.append(fo2)
            elif redoxfunc == "constant0":
                fo2 = redox_constant(planet_age)
                target_fo2.append(fo2)
            elif redoxfunc == "constant2":
                fo2 = redox_constant_2(planet_age)
                target_fo2.append(fo2)
            elif redoxfunc == "inverse":
                fo2 = redox_inverse(planet_age)
                target_fo2.append(fo2)
            CH = CH

            # get mixing ratios through interpolation
            mixingratios = interp([fo2,CH,max(5, min(target_planets[i], 20))])[0]
            #if noise > 0.0:
            #    mixingratios = mixingratios_noise(mixingratios,noise=noise)

            # temp
            # print(planet_age,fo2, CH, target_planets[i])
            # plot_single_piechart(mixingratios)

            # feature scaling
            mixingratios_log = np.log10(np.array(mixingratios)+1e-20)
            mixingratios_log_scaled = scaler.transform(mixingratios_log.reshape(1, -1))
            # assign age bin
            if classifier == "binary":
                if planet_age > agebin_condition:
                    age_class = 'old'
                else:
                    age_class = 'young'
                target_ages.append(age_class)
            if classifier == "continuous":
                target_ages.append(planet_age)
            target_atm_contents.append(np.array(mixingratios))
            target_atm_contents_scaled.append(mixingratios_log_scaled)
        
        if stat_test == "fisherexact":
            classes = []
            young_pop = []
            old_pop = []
            for i in range(len(target_ages)):
                fo2_pred = clf_LR.predict(target_atm_contents_scaled[i].reshape(1, -1))
                classes.append([target_ages[i],fo2_pred])
                if target_ages[i] == "young":
                    young_pop.append(fo2_pred)
                else:
                    old_pop.append(fo2_pred)

            Fishertable = [
                [young_pop.count('oxidized'), len(young_pop) - young_pop.count('oxidized')],
                [old_pop.count('oxidized'),   len(old_pop) - old_pop.count('oxidized')]
            ]

            # print table
            plottable = [
                ["young",young_pop.count('oxidized'), len(young_pop) - young_pop.count('oxidized')],
                ["old",old_pop.count('oxidized'),   len(old_pop) - old_pop.count('oxidized')]
            ]
            headers = ["Population", "Oxidized", "Reduced", "Total"]
            print(tabulate(plottable, headers=headers, tablefmt="grid"))
        
            oddsratio, p_value = fisher_exact(Fishertable)
            if p_value < 0.05:
                print("succes")
                succes_count += 1
            else:
                print("fail")
                fail_count += 1

        if stat_test == "bayesianridge":
            target_fo2_predictions = []
            target_fo2_std= []
            for i in range(len(target_ages)):
                fo2_pred = clf_ridge.predict(target_atm_contents[i].reshape(1, -1),return_std=True)
                target_fo2_predictions.append(fo2_pred[0][0])
                target_fo2_std.append(fo2_pred[1][0])

            # fit data to function:
            def fit_linear(x,a,b):
                return a*x+b
            
            def fit_inverse(x,a,b):
                return (-1/(a*x))+b
            
            fitting = curve_fit(fit_linear,target_ages,target_fo2_predictions,sigma=target_fo2_std)
            if abs(fitting[0][0]) > 3 * np.sqrt(fitting[1][0][0]):
                succes_count += 1
            else:
                fail_count += 1

            X = np.linspace(0,8,1000)
            fig = figure(figsize=(8,6))
            frame = fig.add_subplot(1,1,1)
            frame.errorbar(target_ages,target_fo2_predictions,yerr=target_fo2_std,color='red',fmt='o',label='predicted')
            frame.scatter(target_ages,target_fo2,color='blue',label='real')
            frame.set_ylabel("fo2")
            frame.set_xlabel("stellar age")
            frame.set_xlim(0,8)
            frame.set_ylim(-6,6)
            frame.plot(X,fit_linear(X,fitting[0][0],fitting[0][1]),color="green")
            frame.legend()
            show()

    print("Done. Succes rate = ", succes_count/(succes_count+fail_count))
    return [succes_count,fail_count,succes_count/(succes_count+fail_count)]


In [66]:
LIFE_survey(N_iterations = 100, CH=0.1,redoxfunc="linear",noise=0.0,stat_test="fisherexact",classifier="binary")


+--------------+------------+-----------+
| Population   |   Oxidized |   Reduced |
+==============+============+===========+
| young        |         40 |        10 |
+--------------+------------+-----------+
| old          |         42 |         8 |
+--------------+------------+-----------+
fail
+--------------+------------+-----------+
| Population   |   Oxidized |   Reduced |
+==============+============+===========+
| young        |         34 |        16 |
+--------------+------------+-----------+
| old          |         39 |        11 |
+--------------+------------+-----------+
fail
+--------------+------------+-----------+
| Population   |   Oxidized |   Reduced |
+==============+============+===========+
| young        |         31 |        12 |
+--------------+------------+-----------+
| old          |         52 |         5 |
+--------------+------------+-----------+
succes
+--------------+------------+-----------+
| Population   |   Oxidized |   Reduced |
+==============+=

[50, 50, 0.5]

In [ ]:
# obtained succes rates per noise value (no feature scaling)
noises = [0.0,2.5,5.0,7.5,10.0,15.0]
succesrates = [1.0,0.92,0.71,0.4,0.21,0.23]

# obtained succes rates per noise value (with feature scaling)
succesrates_featurescaled = [1.0,1.0,0.99,0.7,0.3,0.03]

#
succesrates_cont = [1.0,0.9,0.76,0.53,0.58,0.4]

fig = figure(figsize=(8,6))
frame = fig.add_subplot(1,1,1)
frame.set_xlabel("Logarithmic noise")
frame.set_ylabel("Succes rate of distinction between young and old populations")
frame.plot(noises,succesrates,label="Binary, no feature scaling")
frame.plot(noises,succesrates_featurescaled,label="Binary, with feature scaling")
frame.plot(noises, succesrates_cont,label="Continuous, no feature scaling")
frame.legend()
